# Manuscript model tables and event trajectories

Assemble the current manuscript and SI tables from saved or newly reproduced analysis outputs.

**Step:** 11. Assemble the model result tables (after the R scripts 08-10).

**Article scope:** the main article and Supplementary Material.

**Inputs:** Event results, observed means, analysis rows, matched log results, and verified placebo ranks.

**Outputs:** Tables 2, 3, S2, S3, S5, S6 and the Figure 3 trajectory data, with unrounded numeric values.

**How to use this notebook:** select the Python environment described in `README.md`, restart the kernel,
and run the cells from top to bottom. Edit the configuration cell before running.
The checks below stop execution when the current data no longer match the manuscript specification.

## Locate the code folder and load the shared functions

The repository is found by searching this notebook's folder and its parent folders, so Jupyter can be started anywhere inside the repository. Set `INDOOR_AQ_CODE` only when it starts elsewhere.
It loads only the cells tagged `library` in `00_shared_functions.ipynb` (its demonstration cell is not executed).
Each notebook has its own imports and configuration, so it never depends on another notebook's kernel state.

In [ ]:
# Setup: find the code folder and load the shared functions.
from pathlib import Path
import json
import os
import sys
import types


def locate_code_directory():
    """Locate this repository (the folder with 00_shared_functions.ipynb) from the working directory or INDOOR_AQ_CODE."""
    configured = os.environ.get("INDOOR_AQ_CODE")
    if configured:
        candidates = [Path(configured).expanduser().resolve()]
    else:
        candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if ((candidate / "00_shared_functions.ipynb").is_file()
                and (candidate / "5_did_cits").is_dir()):
            return candidate.resolve()
    raise FileNotFoundError("Start Jupyter in the repository folder, or set INDOOR_AQ_CODE to that folder.")

CODE = locate_code_directory()

# Load the cells tagged "library" from 00_shared_functions.ipynb as an in-memory
# module named `shared`, so that `from shared import ...` works below.
# No helper .py file is generated.
_library_path = CODE / "00_shared_functions.ipynb"
_library = types.ModuleType("shared")
_library.__file__ = str(_library_path)
_library.__dict__["CODE"] = CODE
for _cell in json.loads(_library_path.read_text(encoding="utf-8"))["cells"]:
    if _cell["cell_type"] == "code" and "library" in _cell.get("metadata", {}).get("tags", []):
        exec(compile("".join(_cell["source"]), str(_library_path), "exec"), _library.__dict__)
sys.modules["shared"] = _library

from shared import DATA, output_root
print(f"Manuscript code: {CODE}")

## Configure this reproduction run

Choose the separate output directory. The default is `results/` in the repository folder; rebuilt files in that directory take precedence over supplied inputs. Use a new directory when testing a changed input.

In [ ]:
# Paths are shared by Python and R. Defaults: data/ for supplied inputs,
# results/ for this run. Set INDOOR_AQ_DATA and INDOOR_AQ_OUTPUT before
# starting Jupyter and R; relative paths are resolved against the repository.
OUTPUT = os.environ.get("INDOOR_AQ_OUTPUT") or None

## Import the required libraries

These imports belong to the active notebook kernel. Install `requirements.txt` in that same environment if an import fails.

In [ ]:
import pandas as pd
import numpy as np
from shared import CODE, DATA, EVENTS, LABELS, METRICS, OUTCOMES, source_file, output_root

## Read this run's model and placebo outputs

Read the results generated by steps 08, 09, and 10. Reconstruct each empirical tail probability from its integer rank and denominator before formatting the article tables. Missing preceding outputs stop this step.

In [ ]:
out = output_root(OUTPUT)
tables_dir = out / 'tables'

# Read model results from this run only.
results = pd.read_csv(source_file('tables/event_results.csv', out))
events = pd.read_csv(DATA / 'events.csv').set_index('event')
means = pd.read_csv(source_file('tables/home_event_observed_means.csv', out))
rows = pd.read_csv(source_file('data/analysis_rows.csv.gz', out), parse_dates=['date'])

paired_path = source_file('tables/paired_PM25_results.csv', out)  # step 09
logs = pd.read_csv(paired_path)
placebo_path = source_file('tables/placebo_ranks_all_56_primary.csv', out)  # step 10
placebo = pd.read_csv(placebo_path)

# Use the exact empirical fraction from its integer numerator and denominator.
denominator = placebo.finite_refits if 'finite_refits' in placebo else placebo.G
assert np.allclose(placebo.p, placebo['rank'] / denominator, atol=5.1e-5, rtol=0)
placebo['p'] = placebo['rank'] / denominator

## Define helper functions

`result` returns exactly one row for each event-outcome-model-analysis combination. `home_numbers` sorts home
numbers numerically for Table S2 (alphabetical order would put Home 10 before Home 2). `trajectory_frames` builds
the Figure 3 series for one event and outcome; the method is described in the next section.

In [ ]:
def result(event, v, model, analysis='primary'):
    """The single row of event_results.csv for one event, outcome, model and analysis."""
    a = results[(results.event == event) & (results.outcome == v)
                & (results.model == model) & (results.analysis == analysis)]
    assert len(a) == 1
    return a.iloc[0]


def home_numbers(labels):
    """['Home 10', 'Home 2'] -> '2, 10' (numeric, not alphabetical, order)."""
    return ', '.join(str(n) for n in sorted(int(s.split()[-1]) for s in labels))


def trajectory_frames(event, v, e):
    """Figure 3 data for one event and outcome.

    Each home's daily values are centered on its own baseline mean. The
    treated home and the mean of the eligible comparison homes are returned
    as daily values and as a centered 7-day moving average (display only,
    at least 3 observations), computed separately before and after the
    works. Construction days and missing days stay missing.
    """
    z = rows[rows.event.eq(event) & rows.outcome.eq(v) & rows.analysis.eq('primary')]
    start = pd.Timestamp(e.start)
    end = pd.Timestamp(e.end)
    dates = pd.date_range(start - pd.Timedelta(days=90), end + pd.Timedelta(days=30))
    wide = z.pivot(index='date', columns='home', values='y').reindex(dates)
    wide.loc[(wide.index >= start) & (wide.index <= end)] = np.nan      # construction days
    centered = wide - wide.loc[wide.index < start].mean()               # minus own baseline mean
    controls = [h for h in wide.columns if h != e.home]
    series = [('Treated home', centered[e.home]),
              ('Eligible comparisons', centered[controls].mean(axis=1))]

    frames = []
    for label, values in series:
        smooth = pd.Series(np.nan, index=dates)
        for mask in [dates < start, dates > end]:
            part = values.loc[mask]
            smooth.loc[mask] = part.rolling(7, center=True, min_periods=3).mean().where(part.notna())
        frames.append(pd.DataFrame(dict(event=event, outcome=v, series=label, date=dates,
                                        centered_daily_value=values.values, smoothed_value=smooth.values)))
    return frames

## Build event tables, sensitivity comparisons, and trajectories

Tables 2 and S2 describe coverage and comparisons; Table 3 combines treated observed means with DiD/CITS estimates. Table S5 averages comparison-home period means with equal home weights. Figure 3 subtracts each home's own baseline mean, averages eligible comparisons by date, and applies a centered seven-day display smoother separately before and after works (minimum three observations). Missing days and construction dates stay missing. Table S3 retains original-scale and log-scale values in their distinct units; Table S6 retains exact integer ranks.

In [ ]:
table2, table3, s2, s3, s5, s6, trajectories = [], [], [], [], [], [], []

for event in EVENTS:
    e = events.loc[event]
    temp = result(event, 'temp', 'DiD')
    pm = result(event, 'pm25', 'DiD')
    allpm = result(event, 'pm25', 'DiD', 'SI_original_comparisons')

    # Table 2: event dates, treated-home coverage and number of comparison homes.
    table2.append(dict(event=event, home=e.home, start=e.start, end=e.end,
                       baseline_valid_days=temp.pre_n, followup_valid_days=temp.post_n,
                       comparisons_TRHCO2=temp.n_controls, comparisons_PM25=pm.n_controls))

    # Table S2: comparison homes, and the homes excluded by the PM2.5 baseline limit.
    primary_homes = pm.control_homes.split(';')
    all_homes = allpm.control_homes.split(';')
    s2.append(dict(event=event,
                   TRHCO2=home_numbers(temp.control_homes.split(';')),
                   PM25_primary=home_numbers(primary_homes),
                   PM25_excluded=home_numbers(set(all_homes) - set(primary_homes))))

    for v in OUTCOMES:
        did = result(event, v, 'DiD')
        cits = result(event, v, 'CITS')
        obs = means[means.event.eq(event) & means.outcome.eq(v) & means.analysis.eq('primary')]

        # Table 3: treated-home observed means and both model estimates.
        treated = obs[obs.treated].set_index('period')
        table3.append(dict(event=event, outcome=v,
                           before=treated.loc['pre', 'mean'], after=treated.loc['post', 'mean'],
                           DiD_estimate=did.estimate, DiD_lower=did.lower, DiD_upper=did.upper,
                           CITS_estimate=cits.estimate, CITS_lower=cits.lower, CITS_upper=cits.upper))

        # Table S5: comparison-home period means, each home weighted equally.
        comp = obs[~obs.treated].groupby('period')['mean'].mean()
        s5.append(dict(event=event, outcome=v, n_controls=did.n_controls,
                       before=comp['pre'], after=comp['post'], change=comp['post'] - comp['pre']))

        # Table S6: placebo ranks and empirical tail probabilities.
        p = placebo[placebo.event.eq(event) & placebo.outcome.eq(v)].set_index('model')
        assert len(p) == 2 and p.G.nunique() == 1
        s6.append(dict(event=event, outcome=v, G=int(p.G.iloc[0]),
                       DiD_rank=int(p.loc['DiD', 'rank']), DiD_tail=p.loc['DiD', 'p'],
                       CITS_rank=int(p.loc['CITS', 'rank']), CITS_tail=p.loc['CITS', 'p']))

        # Figure 3 trajectories.
        trajectories.extend(trajectory_frames(event, v, e))

    # Table S3: PM2.5 with the primary pool, the expanded pool and on the log1p scale.
    for model in ['DiD', 'CITS']:
        primary = result(event, 'pm25', model)
        expanded = result(event, 'pm25', model, 'SI_original_comparisons')
        log_row = logs[logs.event.eq(event) & logs.model.eq(model)]
        assert len(log_row) == 1
        log_row = log_row.iloc[0]
        s3.append(dict(event=event, model=model, n_primary=primary.n_controls, n_all=expanded.n_controls,
                       primary_estimate=primary.estimate, primary_lower=primary.lower, primary_upper=primary.upper,
                       all_estimate=expanded.estimate, all_lower=expanded.lower, all_upper=expanded.upper,
                       log_estimate=log_row.log1p_estimate, log_lower=log_row.log1p_lower,
                       log_upper=log_row.log1p_upper))

## Save numeric tables and figure inputs

CSV values are not rounded to the manuscript's display precision. Figure 4 and Figure S2 read the same event-results table, and Figure 3 reads the newly assembled trajectories. This step assembles results; it does not estimate a new model.

In [ ]:
for name, values in [('Table_2', table2), ('Table_3', table3), ('Table_S2', s2),
                     ('Table_S3', s3), ('Table_S5', s5), ('Table_S6', s6)]:
    pd.DataFrame(values).to_csv(tables_dir / (name + '.csv'), index=False)
pd.concat(trajectories, ignore_index=True).to_csv(tables_dir / 'trajectory_plot_data.csv', index=False)

# Keep the event metadata alongside the generated table and figure inputs.
(out / 'data').mkdir(exist_ok=True)
events.reset_index().to_csv(out / 'data/events.csv', index=False)
print('Tables 2, 3, S2, S3, S5, S6 and Figure 3 trajectories generated from the current results.')

## Interpret the outputs

Review the saved CSV files and JSON checks in the selected output directory. Keep the settings in `analysis_settings.json`
fixed when reproducing this manuscript. If a source-data assertion fails, inspect the named records before changing
any thresholds. `README.md` documents the order of all manuscript stages.